# Praetor — a walkthrough you can read without running anything

**Praetor is a post-detection disposition-policy engine.** A detection has already fired;
Praetor decides *what should happen next*. It takes the alert + nearby local telemetry,
asks an LLM for a **structured proposal**, then runs **deterministic safety gates** before
emitting a final disposition and a tamper-evident audit record.

> The intelligence is allowed to be non-deterministic; the **authority to act is not.**

Praetor has **no UI and no server** — that is deliberate (an explicit non-goal). It is a
decision engine. So "seeing it in action" means watching one fired alert become a
`DecisionEdict` — and watching the gates refuse unsafe containment. This notebook drives the
**real engine** (`process_alert_intake`) end-to-end. The outputs below are committed, so you
can read the whole flow on GitHub without installing or running anything.

The three dispositions:

| Disposition | Meaning | Fails… |
|---|---|---|
| `standard_review` | route to normal human review (the safe floor) | safe (a human still sees it) |
| `escalate` | route to prioritized human review | loud |
| `auto_contain` | emit a bounded containment directive *before* human review | loud + reviewable |

We will run three alerts:
1. a **confirmed-malicious** office-macro chain -> `auto_contain` + a containment directive,
2. a **benign** interactive logon -> `standard_review`,
3. an `auto_contain` proposed on a **never-contain** host (a domain controller) -> the gate
   **refuses** and escalates with `never_contain_live_conflict`.


### Imports + locate the repo (so `configs/example_org.yaml` is found wherever this runs)

In [1]:
import tempfile
from datetime import UTC, datetime
from pathlib import Path

from praetor.auth.principal import Principal
from praetor.auth.verifier import PrincipalMapVerifier
from praetor.config.activation import activate_org_config
from praetor.config.emergency import add_emergency_never_contain
from praetor.config.state import fetch_outstanding_unrevoked_directives
from praetor.contracts.disposition import Disposition
from praetor.contracts.evidence import EvidenceBundle, EvidenceFact
from praetor.contracts.judgment import CitedEvidenceRef, ModelJudgment
from praetor.engine import process_alert_intake
from praetor.judgment.provider import JudgmentRequest, ProviderProbeResult
from praetor.state.store import open_state_store
from praetor.tickets.stamp import StampBackendOutcome, StampBackendResult


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "configs" / "example_org.yaml").exists():
            return cand
    raise RuntimeError("run this notebook from inside the Praetor repo")


REPO = find_repo_root()
print("Praetor repo:", REPO)

Praetor repo: C:\Users\oalan\Praetor


### Narration glue (the only non-engine code here)

A real deployment supplies a Vertex/Gemini provider and a real ticket system. To keep this
notebook **deterministic and offline**, we script the model's judgment and a succeeding ticket
stamp. **Everything downstream of these two stand-ins is the real engine** — correlation hash,
PolicyGate, never-contain checks, directive construction, the hash-chained ledger.

In [2]:
class ScriptedProvider:
    """Stands in for the LLM: returns a fixed ModelJudgment (the model's proposal)."""

    def __init__(self, judgment: ModelJudgment) -> None:
        self.judgment = judgment

    def generate_judgment(self, request: JudgmentRequest) -> ModelJudgment:
        return self.judgment

    def probe(self, canary_payload):
        return ProviderProbeResult(
            success=True, provider_name="demo", model_name="demo-model"
        )


class SucceedingStamp:
    """Stands in for the ticket system: every stamp succeeds (idempotent)."""

    def stamp(self, stamp_id: str, payload: dict) -> StampBackendResult:
        return StampBackendResult(
            outcome=StampBackendOutcome.SUCCEEDED, payload={"ticket_id": "INC-DEMO-001"}
        )


def host_evidence(
    host_id: str, evidence_id: str, process: str, parent: str
) -> EvidenceBundle:
    """Sysmon + security facts so host auto_contain clears the floor (DEC-059)."""
    ts = datetime.now(UTC)
    return EvidenceBundle(facts=[
        EvidenceFact(
            evidence_id=evidence_id,
            normalized_fields={
                "host_id": host_id,
                "process_name": process,
                "parent_process_name": parent,
            },
            source_event_reference=f"sysmon:{host_id}:1",
            raw_source=(
                '{"_comment": "local-only, structurally excluded '
                'from the model prompt"}'
            ),
            provenance_path="sysmon_event_log",
            ambiguity_flag=False,
            timestamp=ts,
        ),
        EvidenceFact(
            evidence_id=f"{evidence_id}-sec",
            normalized_fields={"host_id": host_id, "event_id": 4624},
            source_event_reference=f"security:{host_id}:1",
            raw_source='{"_comment": "corroborating security event"}',
            provenance_path="windows_security_log",
            ambiguity_flag=False,
            timestamp=ts,
        ),
    ])


def model_proposes(disposition, evidence_id, narrative, tells, *, corroborated=False):
    """The structured judgment the model emits. Citations resolve to bundle facts."""
    refs = [CitedEvidenceRef(evidence_id=evidence_id, field_path="host_id")]
    if corroborated:
        refs.append(
            CitedEvidenceRef(evidence_id=f"{evidence_id}-sec", field_path="host_id")
        )
    return ModelJudgment(
        proposed_disposition=disposition,
        cited_evidence_refs=refs,
        key_tells=tells,
        org_config_refs=["containment_policy.default_action"],
        benign_alternatives=["scheduled IT automation"],
        benign_alternatives_ruled_out=["no change ticket; off-hours; encoded payload"],
        convergence_reasoning=(
            "office app spawning encoded PowerShell matches the intrusion pattern"
        ),
        narrative=narrative,
        model_name="demo-model",
        provider_name="demo",
    )


def run_case(store, *, alert_id, bundle, judgment):
    """Drive ONE alert through the real engine; print the decision + directive."""
    result = process_alert_intake(
        store,
        judgment_provider=ScriptedProvider(judgment),
        stamp_backend=SucceedingStamp(),
        alert_identity=alert_id,
        evidence_bundle=bundle,
    )
    e = result.edict
    print(f"alert             : {alert_id}")
    print(f"model proposed    : {judgment.proposed_disposition.value}")
    print(f"PRAETOR DECIDED   : {e.final_disposition.value.upper()}")
    print(f"fault_flags       : {e.fault_flags or '[]'}")
    print(f"system_fault_esc. : {e.system_fault_escalation}")
    print(f"stamp_status      : {e.stamp_status}")
    print(f"decision_id       : {e.decision_id}")
    print(f"ledger_curr_hash  : {e.ledger_current_hash}")
    host = bundle.facts[0].normalized_fields["host_id"]
    dirs = [
        d
        for d in fetch_outstanding_unrevoked_directives(store.conn)
        if d.target_id == host
    ]
    if dirs:
        d = dirs[0]
        print("  >> CONTAINMENT DIRECTIVE EMITTED")
        print(
            f"     target          : {d.target_type.value}:{d.target_id}  "
            f"scope={d.scope}"
        )
        lifetime_s = (d.expires_at - d.issued_at).total_seconds()
        print(f"     lifetime        : {lifetime_s:.0f}s  (hard cap 300)")
        print(f"     status          : {d.status.value}")
        print(f"     idempotency_key : {d.idempotency_key}")
        print(f"     live_nc_hash    : {d.live_never_contain_hash}")
        print(f"     evidence_refs   : {d.evidence_refs}")
    else:
        print("  >> no containment directive — nothing isolated")
    return result

### Boot Praetor: open a throwaway DB and activate the org config

The org config is the human-authored statute (`configs/example_org.yaml`): asset registry,
never-contain lists, containment policy, rate limits, breaker thresholds, directive-lifetime
cap. Activation runs preflight and binds a snapshot hash.

In [3]:
from praetor.config.snapshot import compute_snapshot_hash_from_binding
from praetor.config.state import fetch_active_snapshot, persist_org_config_snapshot
from praetor.contracts.org_config_sections import ContainmentPolicy, ContainmentRule


def allow_host_containment(store, host_id: str) -> None:
    """Progressive authorization: explicit scoped allow under escalate default."""
    base = fetch_active_snapshot(store.conn)
    assert base is not None
    policy = ContainmentPolicy(
        default_action="escalate",
        rules=[
            ContainmentRule(
                name=f"walkthrough_allow_{host_id}",
                action="allow",
                scope={"target_type": "host", "target_id": host_id},
            )
        ],
    )
    payload = base.model_dump(mode="json")
    payload["containment_policy"] = policy.model_dump(mode="json")
    payload.pop("snapshot_hash", None)
    snapshot_hash = compute_snapshot_hash_from_binding(payload)
    updated = base.model_copy(
        update={"containment_policy": policy, "snapshot_hash": snapshot_hash}
    )
    persist_org_config_snapshot(
        store.conn, updated, verbatim_render_text="walkthrough"
    )
    store.conn.execute(
        """
        UPDATE active_org_config
        SET snapshot_hash = ?, verbatim_render_id = ?
        WHERE id = 1
        """,
        (updated.snapshot_hash, "walkthrough-render"),
    )
    store.conn.commit()


TOKEN = "soc-lead-token"
VERIFIER = PrincipalMapVerifier(
    {TOKEN: Principal(identity="soc-lead-1", role="soc_lead")}
)

_tmp = tempfile.TemporaryDirectory(prefix="praetor-walkthrough-")
store = open_state_store(Path(_tmp.name) / "walkthrough.db")
activate_org_config(
    store, REPO / "configs" / "example_org.yaml", token=TOKEN, verifier=VERIFIER
)
allow_host_containment(store, "WORKSTATION1")
print("activated configs/example_org.yaml into a throwaway SQLite state store")
print("bound explicit allow rule for WORKSTATION1 (escalate-by-default posture)")

activated configs/example_org.yaml into a throwaway SQLite state store
bound explicit allow rule for WORKSTATION1 (escalate-by-default posture)


## Case 1 — confirmed malicious -> `auto_contain`

`winword.exe` spawned an encoded PowerShell child on `WORKSTATION1`. The model proposes
`auto_contain`; every deterministic gate passes (citations resolve, host not on never-contain,
rate limit OK, breakers closed, feed healthy), the ticket stamp succeeds — so Praetor emits a
**bounded containment directive** alongside the audit edict.

In [4]:
_ = run_case(
    store,
    alert_id="ALERT-MALICIOUS-001",
    bundle=host_evidence("WORKSTATION1", "ev-mal-1", "powershell.exe", "winword.exe"),
    judgment=model_proposes(
        Disposition.AUTO_CONTAIN,
        "ev-mal-1",
        "winword.exe spawned an encoded PowerShell child on WORKSTATION1.",
        ["winword.exe -> powershell.exe -enc", "off-hours execution"],
        corroborated=True,
    ),
)

alert             : ALERT-MALICIOUS-001
model proposed    : auto_contain
PRAETOR DECIDED   : AUTO_CONTAIN
fault_flags       : []
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : b6911f41f30e86f627a6fd915901fcb87b67f770772ca89e4ad0af17bb8086bd
ledger_curr_hash  : 05850cea1e9d1a8c54799b9a2a2c765fcf1b0a847c49d6c13f0c5f63ab397ace
  >> CONTAINMENT DIRECTIVE EMITTED
     target          : host:WORKSTATION1  scope=host-isolation
     lifetime        : 300s  (hard cap 300)
     status          : emitted
     idempotency_key : 10c19e6300b7db40e58dd38e7b34fa420f73f707a68c0c09530d4ae7060002e1
     live_nc_hash    : 4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba873c2f11161202b945
     evidence_refs   : ['ev-mal-1', 'ev-mal-1-sec']


Note the directive: **scope `host-isolation`, exactly 300s lifetime** (the hard cap), an
idempotency key (so a re-judged alert will not double-isolate the host), and `live_nc_hash` (the
never-contain snapshot the consumer re-verifies before acting). Praetor does **not** call EDR —
it emits an honest, short-lived, revocable directive; the downstream consumer owns actuation.

## Case 2 — benign admin activity -> `standard_review`

A routine interactive logon shell on `WORKSTATION7`. The model proposes `standard_review`; the
gate agrees. No directive — a human still sees it. This is the safe floor.

In [5]:
_ = run_case(
    store,
    alert_id="ALERT-BENIGN-001",
    bundle=host_evidence("WORKSTATION7", "ev-ben-1", "explorer.exe", "userinit.exe"),
    judgment=model_proposes(
        Disposition.STANDARD_REVIEW,
        "ev-ben-1",
        "Routine interactive logon shell on WORKSTATION7.",
        ["explorer.exe launched by userinit.exe at logon"],
    ),
)

alert             : ALERT-BENIGN-001
model proposed    : standard_review
PRAETOR DECIDED   : STANDARD_REVIEW
fault_flags       : []
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : 62adf945d3342fa89c14665f3733672e4e0d8654c6264305d8f2e9cf880aafbb
ledger_curr_hash  : b949aa16b264ecc94c865d2c23ab22667c051b5fe90f5ce5b8d7da5572f77c2f
  >> no containment directive — nothing isolated


## Case 3 — `auto_contain` proposed on a never-contain host -> the gate refuses

A SOC lead has flagged the domain controller `DC01` as **never-contain** (here via an emergency
entry; permanent entries live in the org config). Now an alert proposes `auto_contain` on `DC01`.
The model can *propose* it — but the deterministic live never-contain check **overrides** the
proposal and escalates with `never_contain_live_conflict`. **Uncertainty flows downward; the
model may not bypass the gates.** Note `system_fault_escalation = False`: this is a deliberate
policy/safety gate firing, not an infrastructure fault.

In [6]:
add_emergency_never_contain(
    store,
    token=TOKEN,
    verifier=VERIFIER,
    target_specification={"target_type": "host", "target_id": "DC01"},
    lifetime_seconds=3600,
    audit_reason="domain controller — never auto-contain",
)
allow_host_containment(store, "DC01")

_ = run_case(
    store,
    alert_id="ALERT-DC-001",
    bundle=host_evidence("DC01", "ev-dc-1", "powershell.exe", "services.exe"),
    judgment=model_proposes(
        Disposition.AUTO_CONTAIN,
        "ev-dc-1",
        "Suspicious PowerShell on domain controller DC01.",
        ["lsass handle access", "encoded command"],
        corroborated=True,
    ),
)

alert             : ALERT-DC-001
model proposed    : auto_contain
PRAETOR DECIDED   : ESCALATE
fault_flags       : ['never_contain_live_conflict']
system_fault_esc. : False
stamp_status      : succeeded
decision_id       : 4429feb463b65f779e629a2e0849fa8271040302c0b589b32b455399063533b5
ledger_curr_hash  : 77f964a220d66e9b65f39add027f0a91e314ad67137bd3100ef731630c3f5bd6
  >> no containment directive — nothing isolated


In [7]:
store.close()
_tmp.cleanup()
print("done — throwaway DB cleaned up")

done — throwaway DB cleaned up


## What you just saw

- The **same `auto_contain` proposal** produced a containment directive on a normal workstation
  (Case 1) and a **refusal** on a never-contain host (Case 3). The model proposes; deterministic
  gates authorize. That separation is the entire product thesis.
- Every decision produced a **`decision_id`** and a **hash-chained `ledger_current_hash`** — a
  tamper-evident audit row, whether or not anything was contained.
- The containment directive is **short-lived (<=300s), idempotent, and carries a never-contain
  snapshot hash** for the consumer to re-verify. Praetor never actuates; it emits honest,
  revocable signals.

### Where to look next
- `python -m evals.harness` — the engine adjudicating **all 26 Outcome-Matrix scenarios**
  (malformed model JSON, provider timeouts, rate limits, breaker-open, feed-unhealthy, …).
- `docs/spec.md` — frozen source of truth · `docs/contracts.md` — hashes, IDs, Outcome Matrix.
- `consumer_sdk/reference_verifier.py` — the consumer-side pre-actuation checks that fail closed
  on a stale, expired, or revoked directive.
